# CSIRO Biomass - Ensemble Training

Search weighted ensembles across available OOF/submission files mounted under `/kaggle/input` and save ensemble diagnostics plus a submission.

In [ ]:
import json, itertools
from pathlib import Path
import numpy as np
import pandas as pd
import torch

DATA_DIR = Path('/kaggle/input/csiro-biomass')
INPUT_ROOT = Path('/kaggle/input')
OUT_DIR = Path('/kaggle/working')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type != 'cuda': raise RuntimeError('GPU is required by experiment policy, although ensemble search is lightweight.')
TARGETS = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
WEIGHTS = np.array([0.1,0.1,0.1,0.2,0.5], dtype=np.float32)

def weighted_r2(y, p):
    scores = [1 - np.sum((y[:, i]-p[:, i])**2) / max(np.sum((y[:, i]-y[:, i].mean())**2), 1e-12) for i in range(len(TARGETS))]
    return float(np.sum(np.array(scores) * WEIGHTS)), dict(zip(TARGETS, scores))

In [ ]:
train_long = pd.read_csv(DATA_DIR / 'train.csv')
truth = train_long.pivot_table(index='image_path', columns='target_name', values='target', aggfunc='first').reset_index()
sample = pd.read_csv(DATA_DIR / 'sample_submission.csv')
test_long = pd.read_csv(DATA_DIR / 'test.csv')

oof_files = sorted(INPUT_ROOT.glob('**/oof_predictions.csv'))
sub_files = sorted(INPUT_ROOT.glob('**/submission.csv'))
print('OOF files:', [str(p) for p in oof_files])
print('Submission files:', [str(p) for p in sub_files])
if len(oof_files) < 2:
    raise RuntimeError('Attach at least two prior Kaggle output datasets containing oof_predictions.csv.')

In [ ]:
models = []
for path in oof_files:
    df = pd.read_csv(path)
    pred_cols = [f'pred_{t}' for t in TARGETS]
    if 'image_path' not in df.columns or not all(c in df.columns for c in pred_cols):
        print('Skipping incompatible OOF', path); continue
    merged = truth[['image_path'] + TARGETS].merge(df[['image_path'] + pred_cols], on='image_path', how='inner')
    y = merged[TARGETS].values.astype(np.float32)
    p = merged[pred_cols].values.astype(np.float32)
    score, detail = weighted_r2(y, p)
    models.append({'path': path, 'name': path.parent.name, 'pred': p, 'score': score, 'detail': detail})
    print(path.parent.name, score)
if len(models) < 2: raise RuntimeError('No compatible OOF model files found.')

In [ ]:
y = truth[TARGETS].values.astype(np.float32)
grid = np.linspace(0, 1, 21)
best = {'score': -999, 'weights': None}
if len(models) == 2:
    candidates = [(w, 1-w) for w in grid]
else:
    candidates = []
    for raw in itertools.product(grid, repeat=len(models)):
        s = sum(raw)
        if abs(s - 1) < 1e-6: candidates.append(raw)
for weights in candidates:
    pred = sum(w * m['pred'] for w, m in zip(weights, models))
    score, detail = weighted_r2(y, pred)
    if score > best['score']:
        best = {'score': score, 'weights': weights, 'detail': detail}
print('Best ensemble', best)
with open(OUT_DIR / 'metrics.json', 'w') as f:
    json.dump({'cv_weighted_r2': best['score'], 'weights': list(best['weights']), 'models': [m['name'] for m in models], 'per_target_r2': best['detail']}, f, indent=2)
pd.DataFrame([{'model': m['name'], 'path': str(m['path']), 'cv': m['score']} for m in models]).to_csv(OUT_DIR / 'ensemble_inputs.csv', index=False)

In [ ]:
subs = []
for m in models:
    candidates = list(m['path'].parent.glob('submission.csv'))
    if not candidates:
        raise RuntimeError(f'Missing submission.csv next to {m["path"]}')
    subs.append(pd.read_csv(candidates[0]).sort_values('sample_id').reset_index(drop=True))
base = subs[0][['sample_id']].copy()
target = sum(w * s.sort_values('sample_id')['target'].values for w, s in zip(best['weights'], subs))
base['target'] = np.clip(target, 0, None)
base.to_csv(OUT_DIR / 'submission.csv', index=False)
display(base.head())